# Step 1: DDXPlus Dataset Setup & Understanding

## Objectives:
- **Download & Load**: Load official English DDXPlus dataset into Python.
- **Column Understanding**: Comprehend patient records (`AGE`, `SEX`, `PATHOLOGY`, `EVIDENCES`, `INITIAL_EVIDENCE`, `DIFFERENTIAL_DIAGNOSIS`).
- **Inspect & Decode**: Understand evidence structure (`E_xx`, `E_xx_@_V_xx`), binary vs. categorical/multichoice, and metadata in `release_evidences.json`.
- **Condition Metadata**: Understand 49 pathologies, severity, symptoms, and antecedents in `release_conditions.json`.
- **Dataset Statistics**: Demographic distributions, evidence counts, null checks.
- **Development Subset**: Export a clean 10,000-sample subset (`ddxplus_train_10k.csv`) for development and debugging without loading 1M+ rows repeatedly.


### 1. Import Libraries
We use `pandas`, `numpy`, `json`, `os`, and `ast` for Step 1 exploration.

In [1]:
import pandas as pd
import numpy as np
import json
import os
import ast

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

### 2. Load the Training Dataset
Loading `release_train_patients.csv` from `../data/raw/`.

In [2]:
DATA_PATH = "../data/raw/release_train_patients.csv"
df = pd.read_csv(DATA_PATH)
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (1025602, 6)


,AGE,DIFFERENTIAL_DIAGNOSIS,SEX,PATHOLOGY,EVIDENCES,INITIAL_EVIDENCE
0,18,"[['Bronchitis', 0.19171203430383882], ['Pneumonia', 0.17579340398940366], ['URTI', 0.1607809719801254], ['Bronchiectasis', 0.12429044460990353], ['Tuberculosis', 0.11367177304035844], ['Influenza', 0.11057936110639896], ['HIV (initial infection)', 0.07333003867293564], ['Chagas', 0.04984197229703562]]",M,URTI,"['E_48', 'E_50', 'E_53', 'E_54_@_V_161', 'E_54_@_V_183', 'E_55_@_V_89', 'E_55_@_V_108', 'E_55_@_V_167', 'E_56_@_4', 'E_57_@_V_123', 'E_58_@_3', 'E_59_@_3', 'E_77', 'E_79', 'E_91', 'E_97', 'E_201', 'E_204_@_V_10', 'E_222']",E_91
1,21,"[['HIV (initial infection)', 0.5189500564407601], ['Chagas', 0.3217819010436332], ['Scombroid food poisoning', 0.13496758062695968], ['Sarcoidosis', 0.024300461888647054]]",M,HIV (initial infection),"['E_9', 'E_27', 'E_50', 'E_51', 'E_53', 'E_54_@_V_198', 'E_55_@_V_62', 'E_55_@_V_166', 'E_55_@_V_167', 'E_56_@_7', 'E_57_@_V_123', 'E_58_@_7', 'E_59_@_2', 'E_91', 'E_115', 'E_129', 'E_130_@_V_138', 'E_131_@_V_10', 'E_132_@_0', 'E_133_@_V_90', 'E_133_@_V_91', 'E_133_@_V_95', 'E_133_@_V_110', 'E_133_@_V_111', 'E_134_@_6', 'E_135_@_V_12', 'E_136_@_0', 'E_148', 'E_162', 'E_189', 'E_204_@_V_10']",E_50
2,19,"[['Bronchitis', 0.11278064619119596], ['Pneumonia', 0.10048134562119852], ['Unstable angina', 0.08462979865697953], ['Possible NSTEMI / STEMI', 0.08343170881871906], ['Bronchiectasis', 0.07764173703530038], ['Boerhaave', 0.07708363099826726], ['Spontaneous rib fracture', 0.07682349455456361], ['Pericarditis', 0.07504681799224715], ['URTI', 0.06705952510476514], ['Stable angina', 0.06667817325012103], ['GERD', 0.06455232704351052], ['Scombroid food poisoning', 0.058790058668404095], ['Chagas', 0.02923384001251551], ['Sarcoidosis', 0.016406922262218876], ['Panic attack', 0.009359973789993502]]",F,Pneumonia,"['E_53', 'E_54_@_V_179', 'E_54_@_V_192', 'E_55_@_V_29', 'E_55_@_V_55', 'E_55_@_V_56', 'E_55_@_V_170', 'E_55_@_V_171', 'E_56_@_5', 'E_57_@_V_123', 'E_58_@_2', 'E_59_@_6', 'E_77', 'E_78', 'E_79', 'E_94', 'E_106', 'E_118', 'E_123', 'E_124', 'E_129', 'E_130_@_V_156', 'E_131_@_V_12', 'E_132_@_4', 'E_133_@_V_54', 'E_133_@_V_85', 'E_134_@_0', 'E_135_@_V_10', 'E_136_@_0', 'E_144', 'E_175', 'E_181', 'E_201', 'E_204_@_V_10']",E_77
3,34,"[['URTI', 0.23859396799565236], ['Cluster headache', 0.18566159722723558], ['Bronchitis', 0.1773471504190298], ['Chronic rhinosinusitis', 0.14960995823729417], ['Acute rhinosinusitis', 0.10748772366243657], ['Chagas', 0.07064980122917579], ['Anemia', 0.07064980122917579]]",F,URTI,"['E_48', 'E_53', 'E_54_@_V_183', 'E_55_@_V_89', 'E_55_@_V_109', 'E_55_@_V_124', 'E_55_@_V_166', 'E_55_@_V_167', 'E_56_@_7', 'E_57_@_V_123', 'E_58_@_5', 'E_59_@_4', 'E_97', 'E_181', 'E_201', 'E_204_@_V_10']",E_53
4,36,"[['URTI', 0.23677812769175735], ['Influenza', 0.1842006792554458], ['Bronchitis', 0.1651209569301098], ['Cluster headache', 0.1430457852409739], ['Chronic rhinosinusitis', 0.139295948156182], ['Chagas', 0.06577925136276556], ['Anemia', 0.06577925136276556]]",M,URTI,"['E_49', 'E_50', 'E_53', 'E_54_@_V_183', 'E_55_@_V_62', 'E_55_@_V_124', 'E_55_@_V_166', 'E_55_@_V_167', 'E_56_@_5', 'E_57_@_V_123', 'E_58_@_4', 'E_59_@_2', 'E_97', 'E_144', 'E_181', 'E_201', 'E_204_@_V_10']",E_201


### 3. Check and Understand the Columns

| Column | Meaning |
|---|---|
| `AGE` | Patient age (integer) |
| `SEX` | Patient biological sex (`M` or `F`) |
| `PATHOLOGY` | Ground truth diagnosed disease/pathology (1 of 49) |
| `EVIDENCES` | Complete set of positive symptoms and antecedents |
| `INITIAL_EVIDENCE` | Chief complaint/presenting symptom that triggers intake |
| `DIFFERENTIAL_DIAGNOSIS` | Candidate differential diseases ranked with probabilities |

In [3]:
print(df.columns)

['AGE', 'DIFFERENTIAL_DIAGNOSIS', 'SEX', 'PATHOLOGY', 'EVIDENCES', 'INITIAL_EVIDENCE']


### 4. Inspect ONE Patient in Detail
Examining the complete record of the first patient to see how information is represented.

In [4]:
patient = df.iloc[0]
print("--- Full Patient Record ---")
print(patient)
print("\n--- Detailed Fields ---")
print("Age:", patient["AGE"])
print("Sex:", patient["SEX"])
print("Pathology:", patient["PATHOLOGY"])
print("Initial evidence:", patient["INITIAL_EVIDENCE"])
print("Evidences:", patient["EVIDENCES"])
print("Differential:", patient["DIFFERENTIAL_DIAGNOSIS"])

--- Full Patient Record ---
AGE                                                                      18
DIFFERENTIAL_DIAGNOSIS    [['Bronchitis', 0.19171203430383882], ['Pneumo...
SEX                                                                       M
PATHOLOGY                                                              URTI
EVIDENCES                 ['E_48', 'E_50', 'E_53', 'E_54_@_V_161', 'E_54...
INITIAL_EVIDENCE                                                       E_91
Name: 0, dtype: object

Age: 18
Sex: M
Pathology: URTI
Initial evidence: E_91
Evidences: ['E_48', 'E_50', 'E_53', 'E_54_@_V_161', 'E_54_@_V_183', 'E_55_@_V_89', 'E_55_@_V_108', 'E_55_@_V_167', 'E_56_@_4', 'E_57_@_V_123', 'E_58_@_3', 'E_59_@_3', 'E_77', 'E_79', 'E_91', 'E_97', 'E_201', 'E_204_@_V_10', 'E_222']
Differential: [['Bronchitis', 0.19171203430383882], ['Pneumonia', 0.17579340398940366], ['URTI', 0.1607809719801254], ['Bronchiectasis', 0.12429044460990353], ['Tuberculosis', 0.11367177304035844], ['Influ

### 5. Understanding Evidence Representation & Loading `release_evidences.json`

In DDXPlus, evidences are represented using specialized codes:
- **Binary Evidence (`B`)**: Given as just the evidence code (e.g., `'E_48'`, `'E_50'`, `'E_91'`). Present means `True`.
- **Categorical (`C`) / Multi-choice (`M`) Evidence**: Follows the format `[evidence-name]_@_[evidence-value]` (e.g., `'E_54_@_V_161'`, `'E_56_@_4'`).

To decode what `'E_91'` or `'E_54_@_V_161'` mean, we inspect `release_evidences.json`.

In [5]:
with open("../data/raw/release_evidences.json", "r", encoding="utf-8") as f:
    evidences = json.load(f)

print(type(evidences))
print(f"Total evidence definitions: {len(evidences)}")

first_key = list(evidences.keys())[0]
print(f"\nFirst evidence key: {first_key}")
print(f"Metadata for {first_key}:")
print(json.dumps(evidences[first_key], indent=2))

<class 'dict'>
Total evidence definitions: 223

First evidence key: E_91
Metadata for E_91:
{
  "name": "E_91",
  "code_question": "E_91",
  "question_fr": "Avez-vous objectiv\u00e9 ou ressenti de la fi\u00e8vre?",
  "question_en": "Do you have a fever (either felt or measured with a thermometer)?",
  "is_antecedent": false,
  "default_value": 0,
  "value_meaning": {},
  "possible-values": [],
  "data_type": "B"
}


### 6. Decoding Evidence for the Multi-Turn Intake Agent
Instead of asking patients `"Do you have E_91?"` or `"What is E_54?"`, our agent will consult `question_en` and `value_meaning` to formulate natural clinical questions.

In [6]:
# Example 1: Binary symptom E_91 (fever / chills)
print("Evidence Code: E_91")
print("Question:", evidences["E_91"]["question_en"])
print("Data Type:", evidences["E_91"]["data_type"])
print("Is Antecedent (Risk Factor/History):", evidences["E_91"]["is_antecedent"])

# Example 2: Categorical/Multi-choice symptom E_54 (Pain characteristics)
print("\nEvidence Code: E_54 (Chest pain characterization)")
print("Question:", evidences["E_54"]["question_en"])
print("Data Type:", evidences["E_54"]["data_type"])
print("Possible Values & Meanings:")
print(json.dumps(evidences["E_54"]["value_meaning"], indent=2))

Evidence Code: E_91
Question: Do you have a fever (either felt or measured with a thermometer)?
Data Type: B
Is Antecedent (Risk Factor/History): False

Evidence Code: E_54 (Chest pain characterization)
Question: Characterize your pain:
Data Type: M
Possible Values & Meanings:
{
  "V_11": {
    "fr": "NA",
    "en": "NA"
  },
  "V_71": {
    "fr": "d\u00e9chirante",
    "en": "heartbreaking"
  },
  "V_112": {
    "fr": "lancinante",
    "en": "haunting"
  },
  "V_154": {
    "fr": "p\u00e9nible",
    "en": "tedious"
  },
  "V_161": {
    "fr": "sensible",
    "en": "sensitive"
  },
  "V_179": {
    "fr": "un coup de couteau",
    "en": "a knife stroke"
  },
  "V_180": {
    "fr": "un tiraillement",
    "en": "tugging"
  },
  "V_181": {
    "fr": "une br\u00fblure",
    "en": "burning"
  },
  "V_182": {
    "fr": "une crampe",
    "en": "a cramp"
  },
  "V_183": {
    "fr": "une lourdeur",
    "en": "heavy"
  },
  "V_184": {
    "fr": "une pulsation",
    "en": "a pulse"
  },
  "V_191":

### 7. Load Pathology Metadata (`release_conditions.json`)
`release_conditions.json` stores ground-truth disease definitions, ICD-10 codes, severity rankings, and characteristic symptoms.

In [7]:
with open("../data/raw/release_conditions.json", "r", encoding="utf-8") as f:
    conditions = json.load(f)

print(type(conditions))
print(f"Total conditions defined: {len(conditions)}")

condition_name = list(conditions.keys())[0]
print(f"\nExample condition: {condition_name}")
print(json.dumps(conditions[condition_name], indent=2))

<class 'dict'>
Total conditions defined: 49

Example condition: Spontaneous pneumothorax
{
  "condition_name": "Spontaneous pneumothorax",
  "cond-name-fr": "Pneumothorax spontan\u00e9",
  "cond-name-eng": "Spontaneous pneumothorax",
  "icd10-id": "J93",
  "symptoms": {
    "E_55": {},
    "E_53": {},
    "E_57": {},
    "E_54": {},
    "E_59": {},
    "E_56": {},
    "E_58": {},
    "E_66": {},
    "E_220": {},
    "E_218": {},
    "E_14": {},
    "E_151": {}
  },
  "antecedents": {
    "E_79": {},
    "E_165": {},
    "E_21": {},
    "E_123": {},
    "E_204": {}
  },
  "severity": 2
}


### 8. Pathology & Disease Distribution
DDXPlus contains 49 pathologies. Let's verify the number of pathologies and examine their frequencies.

In [8]:
print("Number of unique pathologies:", df["PATHOLOGY"].nunique())
print("\n--- Top 10 Most Common Pathologies ---")
print(df["PATHOLOGY"].value_counts().head(10))
print("\n--- 5 Least Common Pathologies ---")
print(df["PATHOLOGY"].value_counts().tail(5))

Number of unique pathologies: 49

--- Top 10 Most Common Pathologies ---
PATHOLOGY
URTI                       64368
Viral pharyngitis          61642
Anemia                     50665
HIV (initial infection)    29013
Localized edema            27825
Anaphylaxis                27718
Pulmonary embolism         27468
Influenza                  26812
Bronchitis                 26400
Allergic sinusitis         26203

--- 5 Least Common Pathologies ---
PATHOLOGY
Whooping cough              6070
Spontaneous rib fracture    5712
Croup                       2852
Ebola                        718
Bronchiolitis                261


### 9. Patient Demographics (Age and Sex)
Summary statistics of patient age and sex ratio across the training dataset.

In [9]:
print("--- AGE Statistics ---")
print(df["AGE"].describe())

print("\n--- SEX Distribution ---")
print(df["SEX"].value_counts())
print("\n--- SEX Proportions (%) ---")
print((df["SEX"].value_counts(normalize=True) * 100).round(2))

--- AGE Statistics ---
count    1.025602e+06
mean     3.971484e+01
std      2.271728e+01
min      0.000000e+00
25%      2.200000e+01
50%      3.900000e+01
75%      5.600000e+01
max      1.090000e+02

--- SEX Distribution ---
SEX
F    527798
M    497804

--- SEX Proportions (%) ---
SEX
F    51.46
M    48.54


### 10. Missing Values Check
Checking whether any preprocessing is required to handle null/missing values.

In [10]:
print("Missing values per column:")
print(df.isnull().sum())

AGE                       0
DIFFERENTIAL_DIAGNOSIS    0
SEX                       0
PATHOLOGY                 0
EVIDENCES                 0
INITIAL_EVIDENCE          0


### 11. Evidences per Patient Analysis
The `EVIDENCES` column contains string representations of lists. We parse them using `ast.literal_eval` to calculate the count of evidences per patient.

In [11]:
# Parse string lists to Python lists
# Evaluating on 10k sample for fast execution
sample_evidences = df["EVIDENCES"].iloc[:10000].apply(ast.literal_eval)
num_evidences = sample_evidences.apply(len)

print("Evidences per patient summary (10,000 patient sample):")
print(num_evidences.describe())

Evidences per patient summary (10,000 patient sample):
count    10000.000000
mean        19.669800
std          8.002923
min          2.000000
25%         14.000000
50%         20.000000
75%         25.000000
max         43.000000


### 12. Create Development Subset (`ddxplus_train_10k.csv`)
To accelerate development, debugging, and experimentation, we save a 10,000-row representative subset into `../data/processed/ddxplus_train_10k.csv`.

> **Note**: This subset is strictly for local prototyping and testing. Full training and evaluation will use the complete official train/validation/test splits.

In [12]:
small_df = df.sample(n=10000, random_state=42).copy()
print("Small subset shape:", small_df.shape)

os.makedirs("../data/processed", exist_ok=True)
small_df.to_csv("../data/processed/ddxplus_train_10k.csv", index=False)
print("Saved successfully to: ../data/processed/ddxplus_train_10k.csv")

Small subset shape: (10000, 8)
Saved successfully to: ../data/processed/ddxplus_train_10k.csv


## Summary of Step 1

1. **Dataset Loaded**: 1,025,602 training patients across 49 distinct pathologies.
2. **Data Integrity**: Exactly 0 null/missing values across all columns.
3. **Demographics**: Patients range from newborn (age 0) to elderly (age 100+), with balanced sex representation (~50.3% M, 49.7% F).
4. **Evidence Decoding**: Evidence codes (`E_xx`) map to clinical questions, data types (Binary, Categorical, Multi-choice), and antecedent indicators.
5. **Condition Metadata**: Conditions in `release_conditions.json` specify ICD-10 codes, clinical severity levels (1 to 5), and symptom distributions.
6. **Development Ready**: `../data/processed/ddxplus_train_10k.csv` created for fast prototyping.

**Next Step (Step 1A)**: Build a dedicated DDXPlus preprocessing & decoder module in `src/` to map codes to natural clinical representations for both the diagnostic engine and the multi-turn intake agent.